# Predict labels with a fine‑tuned checkpoint (statistics or gender)

This notebook loads a fine‑tuned Transformer model from a saved checkpoint and predicts probabilities for the full unlabeled set and a set of conflicting descriptions. Use the TASK switch in the config cell to choose "stat" or "gender".

In [1]:
# Mount drive to import datasets
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Configuration
# Choose task: "stat" for statistics, "gender" for gender classification
TASK = "gender"  # "stat" or "gender"

MODEL_ID = "xlm-roberta-large"  # must match the fine-tuned checkpoint base

# Base folders per task
if TASK == "stat":
    DATA_BASE = "./drive/MyDrive/Colab Notebooks"
    MODELS_BASE = "./drive/MyDrive/Colab Notebooks/models/stat"
    OUT_BASE = "./drive/MyDrive/Colab Notebooks"
    PATH_TO_MINE = f"{DATA_BASE}/stat_to_mine.feather"
    PATH_CONFLICTING = f"{DATA_BASE}/conflicting_descr_stat.feather"
    MODEL_CHECKPOINT = f"{MODELS_BASE}/large-checkpoint-2169"  # adjust to your best checkpoint
    OUT_UNLABELED_FEATHER = f"{OUT_BASE}/unlabeled_predicted_stat.feather"
    OUT_CONFLICTING_FEATHER = f"{OUT_BASE}/conflicting_predicted_stat.feather"
    PROB_COL = "probability_is_statistics"
elif TASK == "gender":
    DATA_BASE = "./drive/MyDrive/Colab Notebooks"
    MODELS_BASE = "./drive/MyDrive/Colab Notebooks/models/gender"
    OUT_BASE = "./drive/MyDrive/Colab Notebooks"
    PATH_TO_MINE = f"{DATA_BASE}/gen_to_mine.feather"
    PATH_CONFLICTING = f"{DATA_BASE}/conflicting_descr_gen.feather"
    MODEL_CHECKPOINT = f"{MODELS_BASE}/large-checkpoint-3064"  # adjust to your best checkpoint
    OUT_UNLABELED_FEATHER = f"{OUT_BASE}/unlabeled_predicted_gen.feather"
    OUT_CONFLICTING_FEATHER = f"{OUT_BASE}/conflicting_predicted_gen.feather"
    PROB_COL = "probability_is_gender"
else:
    raise ValueError("TASK must be 'stat' or 'gender'")

# Inference parameters
BATCH_SIZE = 512  # adjust for GPU memory
MAX_TOKEN_LENGTH = None  # None => model max
SEED = 42

In [3]:
# Setup: install (if needed) and import libs
# If running in Colab, uncomment the pip installs below.
!pip install -U transformers datasets accelerate scikit-learn plotly pyarrow

import os
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 143.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 166.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 165.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 51.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:


In [4]:
# Load data
# Choose label column based on TASK
LABEL_COL = "is_statistics" if TASK == "stat" else "is_gender"

set_to_mine = pd.read_feather(PATH_TO_MINE)
set_to_mine = set_to_mine.dropna(subset=["text_mining_description"])  # safety

unlabeled = set_to_mine[(set_to_mine[LABEL_COL] == False)].copy()
print(f"Unlabeled rows: {len(unlabeled):,}")

conflicting = pd.read_feather(PATH_CONFLICTING)
conflicting = conflicting.dropna(subset=["text_mining_description"]).copy()
print(f"Conflicting rows: {len(conflicting):,}")

Unlabeled rows: 1,924,453
Conflicting rows: 1,472


In [5]:
# Tokenizer and tokenization helpers
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding=True,               # dynamic padding
        truncation=True,
        max_length=MAX_TOKEN_LENGTH
    )

# Build HF datasets
# unlabeled_texts = unlabeled["text_mining_description"].astype(str).tolist()
# conflicting_texts = conflicting["text_mining_description"].astype(str).tolist()

# unlabeled_ds = Dataset.from_dict({"text": unlabeled_texts}).map(tokenize_fn, batched=True)
# conflicting_ds = Dataset.from_dict({"text": conflicting_texts}).map(tokenize_fn, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

**How to Upload a Model File to Google Drive in the OECD environment**
1. Compress `model.safetensors` into `model.zip`.
2. Split `model.zip` into one or more files named `model.zip.0xx` using **7-Zip File Manager** (available from the Software Center).
3. Upload `config.json` and all of the split `model.zip.0xx` files to Google Drive.


In [ ]:
# If running in Colab, uncomment the pip installs below.
!apt-get -qq install p7zip-full

In [ ]:
# Uncomment depending on which model to use.
# !7z x "./drive/MyDrive/Colab Notebooks/models/stat/large-checkpoint-2169/model.zip.001"
!7z x "./drive/MyDrive/Colab Notebooks/models/gender/large-checkpoint-3064/model.zip.001"


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs AMD EPYC 7B12 (830F10),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan ./drive/MyDrive/Colab Notebooks/models/gender/large-checkpoint-3064/                                                                              1 file, 104857600 bytes (100 MiB)

Extracting archive: ./drive/MyDrive/Colab Notebooks/models/gender/large-checkpoint-3064/model.zip.001
  0% 1 Open           --
Path = ./drive/MyDrive/Colab Notebooks/models/gender/large-checkpoint-3064/model.zip.001
Type = Split
Physical Size = 104857600
Volumes = 18
Total Physical Size = 1808236634
----
Path = model.zip
Size = 1808236634
--
Path = model.zip
Type = zip
Physical Size = 1808236634

  0%    

In [ ]:
# Uncomment depending on which model to use.
# !mv ./model.safetensors "./drive/MyDrive/Colab Notebooks/models/stat/large-checkpoint-2169/"
!mv ./model.safetensors "./drive/MyDrive/Colab Notebooks/models/gender/large-checkpoint-3064/"

In [6]:
# Load fine‑tuned model from checkpoint for inference
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)

inference_args = TrainingArguments(
    output_dir="./tmp-results",  # no checkpointing for inference
    per_device_eval_batch_size=BATCH_SIZE,
    do_train=False,
    do_eval=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_strategy="no",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=inference_args,
    # tokenizer=tokenizer,
    processing_class=tokenizer,
    data_collator=collator,
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
# Predict unlabeled
# pred_unlabeled = trainer.predict(unlabeled_ds)
# probs_unlabeled = torch.softmax(torch.tensor(pred_unlabeled.predictions), dim=1)[:, 1].cpu().numpy()

# unlabeled = unlabeled.copy()
# unlabeled[PROB_COL] = probs_unlabeled

# print("Unlabeled prediction done.")
# unlabeled.to_feather(OUT_UNLABELED_FEATHER)
# print(f"Saved: {OUT_UNLABELED_FEATHER}")

In [ ]:
# Predict unlabeled in chunks
CHUNK_SIZE = 300_000
PARTS_DIR = f"{OUT_BASE}/unlabeled_prediction_parts_{TASK}"
os.makedirs(PARTS_DIR, exist_ok=True)

part_files = []

for start in range(0, len(unlabeled), CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, len(unlabeled))
    part_no = start // CHUNK_SIZE

    output_part = f"{PARTS_DIR}/unlabeled_predicted_part_{part_no:03d}.feather"

    # Skip if already done
    if os.path.exists(output_part):
        print(f"Skipping existing part {part_no}: {output_part}")
        part_files.append(output_part)
        continue

    print(f"Processing rows {start:,} to {end:,}...")

    chunk = unlabeled.iloc[start:end].copy()

    chunk_ds = Dataset.from_dict({
        "text": chunk["text_mining_description"].astype(str).tolist()
    }).map(tokenize_fn, batched=True)

    pred = trainer.predict(chunk_ds)

    probs = torch.softmax(
        torch.tensor(pred.predictions),
        dim=1
    )[:, 1].cpu().numpy()

    chunk[PROB_COL] = probs
    chunk.to_feather(output_part)

    part_files.append(output_part)
    print(f"Saved: {output_part}")

print("All unlabeled chunks completed.")

Processing rows 0 to 300,000...


Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Saved: ./drive/MyDrive/Colab Notebooks/unlabeled_prediction_parts_gender/unlabeled_predicted_part_000.feather
Processing rows 300,000 to 600,000...


Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Saved: ./drive/MyDrive/Colab Notebooks/unlabeled_prediction_parts_gender/unlabeled_predicted_part_001.feather
Processing rows 600,000 to 900,000...


...

Saved: ./drive/MyDrive/Colab Notebooks/unlabeled_prediction_parts_gender/unlabeled_predicted_part_004.feather
Processing rows 1,500,000 to 1,800,000...


Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Saved: ./drive/MyDrive/Colab Notebooks/unlabeled_prediction_parts_gender/unlabeled_predicted_part_005.feather
Processing rows 1,800,000 to 1,924,453...


Map:   0%|          | 0/124453 [00:00<?, ? examples/s]

Saved: ./drive/MyDrive/Colab Notebooks/unlabeled_prediction_parts_gender/unlabeled_predicted_part_006.feather
All unlabeled chunks completed.


In [ ]:
# Check the number of rows and average text length in each part
for f in sorted(os.listdir(PARTS_DIR)):
    df = pd.read_feather(os.path.join(PARTS_DIR, f))
    print(
        f,
        len(df),
        df["text_mining_description"].str.len().mean()
    )

unlabeled_predicted_part_000.feather 300000 141.50795666666667
unlabeled_predicted_part_001.feather 300000 221.59393333333333
unlabeled_predicted_part_002.feather 300000 350.98554666666666
unlabeled_predicted_part_003.feather 300000 345.9765566666667
unlabeled_predicted_part_004.feather 300000 383.7893066666667
unlabeled_predicted_part_005.feather 300000 443.42699666666664
unlabeled_predicted_part_006.feather 124453 478.68706258587576


In [ ]:
# Check total rows across all parts
sum(
    len(pd.read_feather(os.path.join(PARTS_DIR, f)))
    for f in os.listdir(PARTS_DIR)
)

1924453

In [11]:
# Combine unlabeled prediction parts
part_files = sorted([
    f"{PARTS_DIR}/{f}"
    for f in os.listdir(PARTS_DIR)
    if f.endswith(".feather")
])

print(f"Combining {len(part_files)} files...")

unlabeled_predicted = pd.concat(
    [pd.read_feather(f) for f in part_files],
    ignore_index=True
)

unlabeled_predicted.to_feather(OUT_UNLABELED_FEATHER)

print(f"Saved combined file: {OUT_UNLABELED_FEATHER}")
print(f"Rows: {len(unlabeled_predicted):,}")

Combining 7 files...
Saved combined file: ./drive/MyDrive/Colab Notebooks/unlabeled_predicted_gen.feather
Rows: 1,924,453


In [ ]:
# Predict conflicting descriptions
# pred_conflicting = trainer.predict(conflicting_ds)
# probs_conflicting = torch.softmax(torch.tensor(pred_conflicting.predictions), dim=1)[:, 1].cpu().numpy()

# conflicting = conflicting.copy()
# conflicting[PROB_COL] = probs_conflicting

# print("Conflicting prediction done.")
# conflicting.to_feather(OUT_CONFLICTING_FEATHER)
# print(f"Saved: {OUT_CONFLICTING_FEATHER}")

In [12]:
# Predict conflicting descriptions
conflicting_ds = Dataset.from_dict({
    "text": conflicting["text_mining_description"].astype(str).tolist()
}).map(tokenize_fn, batched=True)

pred_conflicting = trainer.predict(conflicting_ds)
probs_conflicting = torch.softmax(torch.tensor(pred_conflicting.predictions), dim=1)[:, 1].cpu().numpy()

conflicting = conflicting.copy()
conflicting[PROB_COL] = probs_conflicting

print("Conflicting prediction done.")
conflicting.to_feather(OUT_CONFLICTING_FEATHER)
print(f"Saved: {OUT_CONFLICTING_FEATHER}")

Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

Conflicting prediction done.
Saved: ./drive/MyDrive/Colab Notebooks/conflicting_predicted_gen.feather


In [13]:
# Quick sanity checks
print(unlabeled_predicted[[PROB_COL]].describe())
print(conflicting[[PROB_COL]].describe())

unlabeled_predicted.head(n=3), conflicting.head(n=3)

       probability_is_gender
count           1.924453e+06
mean            3.677790e-02
std             1.764208e-01
min             1.043173e-04
25%             2.726548e-04
50%             4.189484e-04
75%             8.693674e-04
max             9.995618e-01
       probability_is_gender
count            1472.000000
mean                0.884337
std                 0.308736
min                 0.000147
25%                 0.988994
50%                 0.997194
75%                 0.998622
max                 0.999510


(            text_mining_description language gen_keywords gen_acronyms  \
 0  Semi-aggregates: Semi-aggregates       en         None         None   
 1              Semi-aggregates: N/A       en         None         None   
 2      CURRENT IMPORTS FINANCE: N/A       en         None         None   
 
    is_gender  probability_is_gender  
 0      False               0.000430  
 1      False               0.000918  
 2      False               0.000463  ,
          text_mining_description  probability_is_gender
 0     SECTORS NOT SPECIFIED: N/A               0.003714
 1    INDUSTRIAL DEVELOPMENT: N/A               0.001080
 2  AGRICULTURAL DEVELOPMENT: N/A               0.000623)

In [ ]:
# Check if the output files exist and their lengths
import os

print(os.path.exists(OUT_UNLABELED_FEATHER))
print(os.path.exists(OUT_CONFLICTING_FEATHER))

df1 = pd.read_feather(OUT_UNLABELED_FEATHER)
df2 = pd.read_feather(OUT_CONFLICTING_FEATHER)

print(len(df1))
print(len(df2))

True
True
1924453
1472
